In [1]:
import pandas as pd
import numpy as np

In [2]:
file_path='/Users/sambit_03/Desktop/Time Series Influenza/Data/Data.xlsx'
xls = pd.ExcelFile(file_path)

In [3]:
subtype_cols = ['AH1', 'AH1N12009', 'AH3', 'AH5', 'ANOTSUBTYPED', 'BVIC', 'BYAM', 'BNOTDETERMINED']
count_cols = subtype_cols + ['INF_A', 'INF_B', 'INF_ALL', 'INF_NEGATIVE', 'SPEC_RECEIVED_NB', 'SPEC_PROCESSED_NB']

In [4]:
dfs = []
for sheet in xls.sheet_names:
    df_sheet = pd.read_excel(file_path, sheet_name=sheet)
    df_sheet['country'] = sheet
    dfs.append(df_sheet)

df = pd.concat(dfs, ignore_index=True)

In [6]:
df['iso_sdate_parsed'] = pd.to_datetime(df['ISO_SDATE'], format='ISO8601', errors='coerce')
if 'ISO_WEEK' in df.columns:
    df = df.drop(columns=['ISO_WEEK'])

iso_cal = df['iso_sdate_parsed'].dt.isocalendar()
df['year'] = iso_cal.year
df['iso_week'] = iso_cal.week

In [7]:
for col in count_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    else:
        df[col] = np.nan

In [8]:
agg_dict = {col: 'sum' for col in count_cols if col in df.columns}
agg_dict['iso_sdate_parsed'] = 'min'

df_agg = df.groupby(['country', 'year', 'iso_week'], as_index=False).agg(agg_dict)

In [9]:
for col in subtype_cols:
    df_agg[col] = df_agg[col].fillna(0)

# Backfill INF_ALL where missing but components (INF_A, INF_B) exist
mask_reconstruct = df_agg['INF_ALL'].isna() & df_agg['INF_A'].notna() & df_agg['INF_B'].notna()
df_agg.loc[mask_reconstruct, 'INF_ALL'] = df_agg.loc[mask_reconstruct, 'INF_A'] + df_agg.loc[mask_reconstruct, 'INF_B']

In [10]:
grid_list = []
for country, group in df_agg.groupby('country'):
    min_date = group['iso_sdate_parsed'].min()
    max_date = group['iso_sdate_parsed'].max()
    
    full_dates = pd.date_range(start=min_date, end=max_date, freq='W-MON')
    
    grid = pd.DataFrame({'iso_sdate_grid': full_dates})
    iso_grid_cal = grid['iso_sdate_grid'].dt.isocalendar()
    grid['year'] = iso_grid_cal.year
    grid['iso_week'] = iso_grid_cal.week
    grid['country'] = country
    grid_list.append(grid)

full_grid = pd.concat(grid_list, ignore_index=True)
df_panel = pd.merge(full_grid, df_agg, on=['country', 'year', 'iso_week'], how='left')
df_panel['iso_sdate_parsed'] = df_panel['iso_sdate_parsed'].fillna(df_panel['iso_sdate_grid'])
df_panel = df_panel.drop(columns=['iso_sdate_grid'])

In [11]:
df_panel['covid_period'] = df_panel['year'].isin([2020, 2021, 2022])

In [12]:
non_covid_baseline = (
    df_panel[~df_panel['covid_period']]
    .groupby(['country', 'iso_week'])['INF_ALL']
    .quantile(0.90)
    .reset_index()
    .rename(columns={'INF_ALL': 'epidemic_threshold'})
)

df_panel = pd.merge(df_panel, non_covid_baseline, on=['country', 'iso_week'], how='left')
df_panel['is_spike'] = df_panel['INF_ALL'] > df_panel['epidemic_threshold']

In [13]:
df_panel.to_csv('cleaned_weekly_flu_panel.csv', index=False)

In [14]:
df = pd.read_csv('cleaned_weekly_flu_panel.csv')
df_clean = df.dropna().reset_index(drop=True)

In [18]:
df_clean.to_csv('flu_clean.csv', index=False)